# Q-Learning Case Study

Notebook version of `q_learning_case_study.py`.

## Imports And Policy Helpers

Import NumPy, random seeding utilities, and the shared gridworld. The helper functions below convert a Q-table into an epsilon-greedy policy and sample actions from that policy during exploration.


In [1]:
from __future__ import annotations

import random

import numpy as np

from gridworld_case_study import GridWorldCaseStudyEnv, format_policy


def epsilon_greedy_policy_from_q(Q, epsilon):
    num_states, num_actions = Q.shape
    policy = np.full((num_states, num_actions), epsilon / num_actions)

    for s in range(num_states):
        best_action = int(np.argmax(Q[s]))
        policy[s, best_action] += 1.0 - epsilon

    return policy


def sample_action_from_policy(policy, state):
    return int(np.random.choice(policy.shape[1], p=policy[state]))


## Q-Learning Loop

The next section performs temporal-difference control. For each episode it interacts with the environment, applies the Q-learning update using the maximum next-state action value, and gradually decays epsilon so the behavior policy shifts from exploration toward exploitation.


In [1]:
def q_learning(
    env,
    num_episodes=5000,
    alpha=0.1,
    gamma=0.9,
    epsilon=1.0,
    epsilon_decay=0.995,
    epsilon_min=0.01,
    max_steps_per_episode=50,
    seed=7,
):
    random.seed(seed)
    np.random.seed(seed)
    num_states = env.observation_space.n
    num_actions = env.action_space.n
    Q = np.zeros((num_states, num_actions))

    for _ in range(num_episodes):
        state, _ = env.reset()
        done = False

        for _ in range(max_steps_per_episode):
            policy = epsilon_greedy_policy_from_q(Q, epsilon)
            action = sample_action_from_policy(policy, state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            target = reward + gamma * (1 - done) * np.max(Q[next_state])
            Q[state, action] += alpha * (target - Q[state, action])
            state = next_state

            if done:
                break

        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    return Q


def q_learning_with_history(
    env,
    num_episodes=5000,
    alpha=0.1,
    gamma=0.9,
    epsilon=1.0,
    epsilon_decay=0.995,
    epsilon_min=0.01,
    max_steps_per_episode=50,
    seed=7,
    snapshot_episodes=None,
):
    random.seed(seed)
    np.random.seed(seed)
    num_states = env.observation_space.n
    num_actions = env.action_space.n
    Q = np.zeros((num_states, num_actions))
    snapshots = []

    if snapshot_episodes is None:
        snapshot_episodes = [1, 10, 100, 500, 1000, num_episodes]
    snapshot_set = set(snapshot_episodes)

    for episode in range(1, num_episodes + 1):
        state, _ = env.reset()
        done = False

        for _ in range(max_steps_per_episode):
            policy = epsilon_greedy_policy_from_q(Q, epsilon)
            action = sample_action_from_policy(policy, state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            target = reward + gamma * (1 - done) * np.max(Q[next_state])
            Q[state, action] += alpha * (target - Q[state, action])
            state = next_state

            if done:
                break

        if episode in snapshot_set:
            snapshots.append(
                {
                    "episode": episode,
                    "epsilon": epsilon,
                    "Q": Q.copy(),
                    "policy": policy.copy(),
                }
            )

        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    return Q, snapshots


def greedy_policy_from_q(Q):
    num_states, num_actions = Q.shape
    policy = np.zeros((num_states, num_actions))

    for s in range(num_states):
        best_action = int(np.argmax(Q[s]))
        policy[s, best_action] = 1.0

    return policy


## Run The Learner

Create the environment, train the Q-table, and print a sequence of saved snapshots. The final policy is the greedy policy extracted from the learned action values.


In [2]:
env = GridWorldCaseStudyEnv()
Q, snapshots = q_learning_with_history(env)
policy = greedy_policy_from_q(Q)

np.set_printoptions(precision=3, suppress=True)
print("Q-Learning Snapshots")
for snapshot in snapshots:
    print(
        f"\nEpisode {snapshot['episode']} "
        f"(epsilon={snapshot['epsilon']:.3f})"
    )
    print("Q")
    print(snapshot["Q"])
    print("max_a Q(s, a)")
    print(snapshot["Q"].max(axis=1).reshape(3, 3))
    print(f"Epsilon-greedy behavior policy (epsilon={snapshot['epsilon']:.3f})")
    print(format_policy(snapshot["policy"]))

print("Learned action values Q")
print(Q)
print("\nGreedy policy extracted from the learned Q-table")
print(format_policy(policy))


Q-Learning Snapshots

Episode 1 (epsilon=1.000)
Q
[[-0.19  -0.1   -0.1   -0.19 ]
 [-0.19  -0.1   -0.271  0.   ]
 [ 0.     0.     0.     0.   ]
 [ 0.    -0.1   -0.19  -0.1  ]
 [-0.1   -0.271  0.    -0.271]
 [ 0.    -0.271  0.     0.91 ]
 [-0.1   -0.19  -0.1    0.   ]
 [-0.19   0.    -0.1   -0.1  ]
 [ 0.     0.     0.     0.   ]]
max_a Q(s, a)
[[-0.1   0.    0.  ]
 [ 0.    0.    0.91]
 [ 0.    0.    0.  ]]
Epsilon-greedy behavior policy (epsilon=1.000)
+---+---+---+
| ^ | ^ | ^ |
+---+---+---+
| ^ | ^ | ^ |
+---+---+---+
| ^ | ^ | G |
+---+---+---+

Episode 10 (epsilon=0.956)
Q
[[-1.038 -1.055 -1.162 -1.315]
 [-0.658 -0.654 -0.612 -0.682]
 [-0.581 -0.425  0.173 -0.311]
 [-0.951 -0.521 -0.65  -0.803]
 [-0.637  0.016  0.169 -0.808]
 [-0.54  -0.391  3.091  0.02 ]
 [-0.336 -0.158 -0.1   -0.199]
 [-0.343  3.439 -0.1   -0.19 ]
 [ 0.     0.     0.     0.   ]]
max_a Q(s, a)
[[-1.038 -0.612  0.173]
 [-0.521  0.169  3.091]
 [-0.1    3.439  0.   ]]
Epsilon-greedy behavior policy (epsilon=0.956)
+--